# Find suitable algorithms with automl

Previously, we simply used multiple linear regression to learn and generalize from the student performance dataset.
Before deploying the model, we want to ensure we use the best available algorithm.
Instead of programming several different flows ourselves, we will use Azure's AutoML service instead.

See MS-Learn for tutorial and available metrics: https://learn.microsoft.com/en-us/azure/machine-learning/how-to-configure-auto-train?view=azureml-api-2&tabs=python


In [1]:
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input, automl
from azure.ai.ml.constants import AssetTypes

In [2]:
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception as ex:
    credential = InteractiveBrowserCredential()
    
ml_client = MLClient.from_config(credential=credential, path="../config.json")


Found the config file in: ../config.json


In [3]:
student_data = Input(type=AssetTypes.MLTABLE, path="azureml:students-training:1")

We have to look at the dock for the interface

* [RegressionJob](https://learn.microsoft.com/en-us/python/api/azure-ai-ml/azure.ai.ml.automl.regressionjob?view=azure-python)
* 

In [4]:

job = automl.RegressionJob(
    display_name="student_performance-automl",
    experiment_name="student_performance-automl",
    compute="cluster-mlops",
    primary_metric="r2_score"
)

job.set_data(
    training_data=student_data,
    target_column_name="Performance Index"
)

job.set_limits(
    enable_early_termination=True,
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=5
)

job.set_training(
    enable_model_explainability=True,
)


In [5]:
run_ref = ml_client.jobs.create_or_update(job)

print(f"track the run at: { run_ref.studio_url }")

track the run at: https://ml.azure.com/runs/strong_kale_r8m7slkq39?wsid=/subscriptions/34503611-a3d3-454f-806e-1abc48aeba67/resourcegroups/rg-mlops-0a0ad1f0eeee45bf86/workspaces/mlw-mlops-0a0ad1f0eeee45bf86&tid=f5b0ba10-7374-4ab8-ae85-408826951cc0
